# OmniVoice Project Studio — Google Colab

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/binhminhanh1235/OmniVoice/blob/feat/hardware-quality-presets-v2/notebooks/OmniVoice_Project_Studio_Colab.ipynb)

Long-form workflow:

**Text Doctor → Voice Doctor / Auto Best Segment → Hardware & Quality → Project → Preview → Generate / Resume → Section History → Project Queue**

Project Queue can render multiple projects continuously. Queue state is persisted separately, while each project keeps its own `section-status.json`, so a Colab restart resumes from the first unfinished section instead of restarting completed work.


In [ ]:
# Install Project Studio with hardware-aware quality presets
!pip install -q --upgrade "git+https://github.com/binhminhanh1235/OmniVoice.git@feat/hardware-quality-presets-v2"

import torch
from omnivoice.hardware_quality import detect_hardware

print("CUDA:", torch.cuda.is_available())
hardware = detect_hardware()
print(hardware.summary())
for note in hardware.notes:
    print("-", note)
if not torch.cuda.is_available():
    raise RuntimeError("Enable a GPU runtime: Runtime → Change runtime type → T4 GPU")


## Mount Google Drive

Projects, voice prompts, generated WAVs, reference candidates, `section-status.json`, Section History snapshots, queue state (`project-queue.json`), hardware default (`hardware-quality.json`), adaptive verification reports, and diagnostics persist under:

`MyDrive/OmniVoiceStudio/`


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

!mkdir -p "/content/drive/MyDrive/OmniVoiceStudio"


## Launch Project Studio

Recommended flow:

1. **Text Doctor**: clean safe script issues and review warnings.
2. **Voice Doctor**: analyze a short reference or use **Find Best Segments** for a long recording, then save the voice.
3. **Hardware & Quality**: inspect GPU/VRAM and choose `SAFE`, `BALANCED`, or `FAST`. T4/16 GB normally recommends `BALANCED` with ASR kept on CPU. `SAFE` remains the strongest final-render policy.
4. **Project Studio**: create projects, Preview, Generate / Resume, and review Section History. A project can inherit the workspace quality preset or save its own override in `studio.json`.
5. **Project Queue**: add several existing projects in the desired order. The quality-aware controller automatically uses each project's saved preset.
6. Click **Run Queue**. Studio renders one section at a time, one project at a time. Completed sections/projects are skipped automatically.
7. If Colab disconnects, reopen this notebook and click **Run Queue** again. A stale running queue item is recovered as pending, and each project's `section-status.json` resumes only unfinished sections.
8. **Pause after current section** safely stops before the next section. One failed project can optionally be skipped so the remaining queue continues.

Preset behavior:

- `SAFE`: 32 diffusion steps, 3 retries, adaptive retry + pacing guard + ASR verification.
- `BALANCED`: 28 steps, 2 retries, keeps adaptive retry + pacing guard + ASR verification.
- `FAST`: 24 steps, 1 retry, disables adaptive repair/pacing timestamps but **keeps ASR text verification**.


In [ ]:
!omnivoice-project-studio \
  --model k2-fsa/OmniVoice \
  --workspace "/content/drive/MyDrive/OmniVoiceStudio" \
  --asr-model openai/whisper-small.en \
  --asr-device cpu \
  --share
